# MLP 레이어: 트랜스포머의 피드포워드 네트워크 - MLP 레이어의 구조와 역할

- Tutorial ID: `tut-11-1`
- Tutorial: MLP 레이어: 트랜스포머의 피드포워드 네트워크
- Section ID: `s11-1-1`
- Section: MLP 레이어의 구조와 역할


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("MLP 레이어 분석: 키-값 메모리")
print("=" * 55)

def gelu(x):
    """GELU 활성화 함수"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def relu(x):
    return np.maximum(0, x)

d_model = 8
d_ff = 4 * d_model  # 보통 4배

np.random.seed(0)

# MLP 가중치
W_1 = np.random.randn(d_ff, d_model) * 0.3  # 업프로젝션
b_1 = np.zeros(d_ff)
W_2 = np.random.randn(d_model, d_ff) * 0.3  # 다운프로젝션
b_2 = np.zeros(d_model)

print(f"d_model={d_model}, d_ff={d_ff} ({d_ff//d_model}×d_model)")
print(f"W_1: {W_1.shape} (업프로젝션)")
print(f"W_2: {W_2.shape} (다운프로젝션)")
print(f"총 MLP 파라미터: {W_1.size + W_2.size:,}")

In [ ]:
# MLP 전방 패스

In [ ]:
x = np.random.randn(d_model)
print(f"\n입력 x.shape: {x.shape}")

# 단계별 계산
h = W_1 @ x + b_1           # 업프로젝션: (d_ff,)
h_act = gelu(h)              # 활성화: (d_ff,)
out = W_2 @ h_act + b_2      # 다운프로젝션: (d_model,)

print(f"h (업프로젝션): {h.shape}")
print(f"h_act (GELU 후): {h_act.shape}")
print(f"  활성화된 뉴런 수: {np.sum(np.abs(h_act) > 0.1)}/{d_ff}")
print(f"  희소도: {1 - np.sum(np.abs(h_act) > 0.1)/d_ff:.2f}")
print(f"out (다운프로젝션): {out.shape}")

In [ ]:
# 키-값 메모리 관점

In [ ]:
print("\n" + "=" * 55)
print("키-값 메모리 관점")
print("=" * 55)
print("""
MLP 계산 = 뉴런들의 가중합:
    out = Σ_i gelu(k_i · x) · v_i

여기서:
  k_i = W_1[i, :] : 뉴런 i의 "키" 벡터 (입력 패턴)
  v_i = W_2[:, i] : 뉴런 i의 "값" 벡터 (출력 기여)
  gelu(k_i · x) : 뉴런 i의 활성화 강도

해석:
  - 입력 x와 키 k_i가 유사하면 뉴런 활성화
  - 활성화된 뉴런의 값 v_i가 출력에 기여
  - 이것이 "키-값 메모리"!
""")

# 각 뉴런의 활성화 분석
activations = gelu(W_1 @ x)
contributions = W_2 * activations[np.newaxis, :]  # (d_model, d_ff)

print("상위 활성화 뉴런 (기여도 기준):")
neuron_contrib_norms = np.linalg.norm(contributions, axis=0)
top_neurons = np.argsort(neuron_contrib_norms)[-5:][::-1]

for n in top_neurons:
    ki = W_1[n]  # 키
    vi = W_2[:, n]  # 값
    act = activations[n]
    ki_sim = np.dot(ki, x) / (np.linalg.norm(ki) * np.linalg.norm(x) + 1e-8)
    print(f"  뉴런 {n:3d}: 활성화={act:+.3f}, 키-입력 유사도={ki_sim:+.3f}, 기여노름={neuron_contrib_norms[n]:.3f}")

In [ ]:
# GELU vs ReLU 비교

In [ ]:
print("\n" + "=" * 55)
print("활성화 함수 비교")
print("=" * 55)

x_range = np.linspace(-3, 3, 7)
print("x값  |  ReLU  |  GELU")
print("-----|--------|-------")
for xi in x_range:
    print(f"{xi:5.1f} | {relu(xi):6.3f} | {gelu(xi):6.3f}")

print("\nGELU 특성:")
print("  - 음수 입력에 대해 작은 음수 출력 (소프트한 게이팅)")
print("  - x≈-0.17에서 최솟값 (약 -0.09)")
print("  - x>0에서 거의 선형")
print("  - 이것이 '은닉 레이어의 희소 활성화'를 만들어냄")